# Chapter 4 — Discrete Behavior Cloning

Chapter 3 left us with a backbone that can *see* the brick and *read* the
instruction but cannot move the arm. This notebook builds the missing
piece — the **action head** — and trains it by behavior cloning on the
SO-101 pick-and-place demonstrations.

| Section | What you build | Figure |
| --- | --- | --- |
| 4.2 | Why regression collapses, and what a mixture fixes | 4.4 |
| 4.3 | A uniform per-control action tokenizer | — |
| 4.4 | Three action heads: factorized, autoregressive, parallel | — |
| 4.5 | One shared training loop for all three | — |
| 4.6 | What each head actually learned | 4.8, 4.9 |
| 4.7 | Bins back to motor commands, and how to schedule them | 4.10, 4.11 |

**Before you run:** *Runtime → Change runtime type → GPU*. The
autoregressive head decodes 96 positions in series and is very slow on
CPU.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import importlib.util
import subprocess
import sys

# Switch to 'main' after this feature branch is merged.
CHAPTER_4_REF = 'codex/ch04-colab-training-reporting'

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    chapter4_requirement = (
        f'lrm-ch04[data] @ git+https://github.com/{organization}/'
        f'lrm-code-chapter-4.git@{CHAPTER_4_REF}'
    )
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        chapter4_requirement,
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    # lrm-ch04 stays at version 0.1.0 while the branch is under
    # development, so pip may keep an older 0.1.0 wheel in a reused
    # runtime. Refresh only this small package; keep resolved dependencies.
    refresh = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--force-reinstall', '--no-deps', chapter4_requirement],
        text=True, capture_output=True,
    )
    if refresh.returncode:
        detail = '\n'.join(
            part for part in (refresh.stdout, refresh.stderr) if part
        )
        raise RuntimeError(
            f'Chapter 4 branch refresh failed:\n{detail}'
        )
    # LeRobot may upgrade Colab's preinstalled torch without upgrading its
    # optional torchaudio wheel. Transformers detects torchaudio by package
    # presence, then imports the incompatible binary on the way to SigLIP,
    # so `from ch03 import VLABackbone` dies with an undefined-symbol
    # OSError. Probe in a child process so a failed import cannot taint
    # this kernel; Chapters 2-4 use no audio, so remove only a broken wheel.
    audio_installed = importlib.util.find_spec('torchaudio') is not None
    audio_probe = subprocess.run(
        [sys.executable, '-c', 'import torch, torchaudio'],
        text=True, capture_output=True,
    )
    if audio_installed and audio_probe.returncode:
        uninstall = subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '--yes',
             'torchaudio'],
            text=True, capture_output=True,
        )
        if uninstall.returncode:
            detail = '\n'.join(
                part for part in (uninstall.stdout, uninstall.stderr)
                if part
            )
            raise RuntimeError(
                f'Could not remove incompatible torchaudio:\n{detail}'
            )
        print('Removed an ABI-incompatible optional torchaudio wheel.')
    api_probe = subprocess.run(
        [sys.executable, '-c',
         'from ch03 import VLABackbone; '
         'from ch04.cli import build_action_head; '
         'from ch04.decoding import decode_action_chunk, '
         'sample_action_grids'],
        text=True, capture_output=True,
    )
    if api_probe.returncode:
        detail = '\n'.join(
            part for part in (api_probe.stdout, api_probe.stderr) if part
        )
        raise RuntimeError(
            f'Chapter 4 API verification failed:\n{detail}'
        )
    stale = [
        name for name in sys.modules
        if name.split('.')[0] in ('ch02', 'ch03', 'ch04', 'transformers')
    ]
    for name in stale:
        sys.modules.pop(name, None)
    if stale:
        print('Cleared cached chapter modules. If the next cell still '
              'fails, use Runtime > Restart session and run again.')
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

## Setup

Every figure uses thin lines, labelled axes, fixed colours per action
head, and distinct dash patterns that remain readable in grayscale.

In [ ]:
import itertools
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON
from ch04.style import use_manuscript_style

use_manuscript_style()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_ROOT = ('/content/ch04-checkpoints'
                   if os.path.isdir('/content') else './ch04-checkpoints')
TENSORBOARD_ROOT = f'{CHECKPOINT_ROOT}/tensorboard'

print(f'device            : {device}')
print(f'checkpoints       : {CHECKPOINT_ROOT}')
print(f'tensorboard logs  : {TENSORBOARD_ROOT}')
print(f'action grid       : H={ACTION_HORIZON} timesteps x '
      f'D={ACTION_DIM} controls')
print(f'bins per control  : {ACTION_BINS}')
print(f'chunk duration    : {ACTION_HORIZON / 30:.2f} s at 30 Hz')
print(f'loss at init      : ln({ACTION_BINS}) = '
      f'{math.log(ACTION_BINS):.3f} nats/token')

## 4.2 Why not just regress the action?

The instinctive design predicts six numbers and minimizes MSE. But
minimizing squared error is maximum likelihood under a Gaussian with fixed
variance, so the optimum is the **conditional mean** `E[a | s]`.

Below, the expert puts half its mass at −1 and half at +1, and the
observation gives no clue which. MSE averages them and lands in a valley
where the expert has essentially no data. A mixture keeps one component
per mode, with weights tracking how often each was demonstrated.

In [ ]:
from ch04.diagnostics import plot_bimodal_comparison
from ch04.exercises import (make_bimodal_actions, train_gmm_baseline,
                            train_mse_baseline)

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
mixture, gmm_history = train_gmm_baseline()

with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
    log_weights, means, sigmas = mixture(torch.zeros(1, 1))

support = float(((expert_actions - collapsed).abs() < 0.25).float().mean())
print(f'MSE prediction   : {collapsed:+.3f}')
print(f'  expert mass within 0.25 of it: {support:.1%}')
print(f'mixture means    : {[round(v, 3) for v in means[0].tolist()]}')
print(f'mixture weights  : '
      f'{[round(v, 3) for v in log_weights.exp()[0].tolist()]}')
print(f'mixture sigmas   : {[round(v, 3) for v in sigmas[0].tolist()]}')

# Figure 4.4
plot_bimodal_comparison(expert_actions.numpy(), collapsed, mixture=mixture)
plt.show()

## 4.3 Action tokenization

Instead of a mixture we take the categorical route: split each control's
range into `B = 256` bins and train with cross-entropy, exactly the
next-token machinery the language backbone already uses. Each marginal can
then hold several modes at once.

Set each control's bounds to its **1st and 99th percentiles** so outliers
do not stretch the bins and waste resolution. Fit these bounds on
**training episodes only**; the episode-disjoint validation split remains
unseen during quantization.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                       make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
normalized_actions = collect_normalized_actions(train_loader, stats)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
half_bin = float(((tokenizer.hi - tokenizer.lo) / tokenizer.n_bins).max()) / 2

print(f'train episodes      : {len(train_loader.dataset.episodes)}')
print(f'validation episodes : {validation_loader.dataset.episodes}')
print()
print('one command through the full conversion path')
print(f'  normalized : {np.round(example, 4)}')
print(f'  bin ids    : {bins}')
print(f'  decoded    : {np.round(decoded, 4)}')
print(f'  error      : {np.abs(decoded - example).max():.5f} '
      f'(bound: {half_bin:.5f}, half a bin width)')

In [ ]:
# Visualize the quantization the policy will be trained against.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(normalized_actions[:, 0], bins=80, color='#4C72B0', alpha=0.75)
for bound, name in ((tokenizer.lo[0], 'q01'), (tokenizer.hi[0], 'q99')):
    axes[0].axvline(bound, color='#C44E52', ls='--', lw=1.4)
    axes[0].annotate(name, xy=(bound, 0.92), xycoords=('data', 'axes fraction'),
                     ha='center', fontsize=8, color='#C44E52')
axes[0].set(xlabel='normalized value, control 0', ylabel='training frames',
            title='Percentile bounds ignore the tails')

error = np.abs(tokenizer.decode(tokenizer.encode(normalized_actions))
               - normalized_actions)
axes[1].hist(error.ravel(), bins=60, color='#55A868', alpha=0.8)
axes[1].axvline(half_bin, color='#C44E52', ls='--', lw=1.4)
axes[1].annotate('half a bin', xy=(half_bin, 0.92),
                 xycoords=('data', 'axes fraction'), ha='right',
                 fontsize=8, color='#C44E52')
axes[1].set(xlabel='absolute round-trip error (normalized)',
            ylabel='control values',
            title='Quantization error is bounded by construction')
fig.tight_layout()
plt.show()

In [ ]:
# One target is an H x D grid: H future timesteps, D control dimensions.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print(f'target grid                  : {tuple(demo_grid.shape)}')
print(f'parallel action positions    : {ACTION_HORIZON} '
      f'(one per timestep)')
print(f'autoregressive scalar tokens : {ACTION_HORIZON * ACTION_DIM} '
      f'(decoded in series)')
print(f'first two timestep vectors   : {demo_grid[0, :2].tolist()}')

## 4.4 Three ways to factor the grid

All three heads optimize the same `[H, D]` label grid and expose the same
`[H, D, B]` logits. They differ only in what each cell is allowed to
condition on:

| Head | Factorization | Cost |
| --- | --- | --- |
| **Factorized** | every cell from the state alone | one pass; cannot represent any dependency |
| **Autoregressive** | each cell on all earlier cells | exact conditioning; 96 serial steps |
| **Parallel** | H bidirectional slots, one per timestep | one pass; still a product of per-cell marginals |

Create a fresh Chapter 3 backbone for each head. This gives all three
variants the same initialization and training budget.

In [ ]:
from ch03 import VLABackbone
from ch04.cli import build_action_head
from ch04.style import head_label

HEAD_NAMES = ('factorized', 'autoregressive', 'parallel')


def make_policy(name):
    """Return a fresh (backbone, head) pair for one head design."""
    backbone = VLABackbone().to(device)
    head = build_action_head(name, backbone).to(device)
    return backbone, head


for name in HEAD_NAMES:
    print(f'{name:>15} -> {head_label(name)}')

## 4.5 One training loop, three heads

The optimizer uses a larger learning rate for the newly initialized head
than for the pretrained trunk, SigLIP stays frozen, and the loss is
label-smoothed cross-entropy over the **non-padded** grid cells only.

`action_head_logits` absorbs the one difference between the heads — the
autoregressive branch needs the expert grid for teacher forcing — so a
single function trains and evaluates all three. The returned dictionary
keeps the scored batch for the diagnostics that follow.

In [ ]:
from ch04.data import action_targets, prepare_batch
from ch04.decoding import (decode_action_chunk, evaluate_open_loop,
                           evaluation_mode, sample_action_grids)
from ch04.diagnostics import (plot_per_joint_metrics,
                              plot_training_curves, temporal_jitter)
from ch04.so101 import SO101_ACTION_NAMES
from ch04.train import (action_head_logits, held_out_metrics,
                        train_action_head)


def run_head_experiment(name, steps=None, samples=None,
                        eval_batches=None):
    # Defaults come from the configuration cell above, so the run size
    # can be changed in one place without touching this function.
    settings = globals()
    steps = settings['TRAIN_STEPS'] if steps is None else steps
    if samples is None:
        samples = (settings['AR_GRID_SAMPLES']
                   if name == 'autoregressive'
                   else settings['GRID_SAMPLES'])
    if eval_batches is None:
        eval_batches = settings['EVAL_BATCHES']
    mirror_root = settings.get('DRIVE_CHECKPOINT_ROOT')
    mirror_dir = (os.path.join(mirror_root, name)
                  if mirror_root is not None else None)
    resume_path = None
    if settings.get('RESUME_FROM_GOOGLE_DRIVE'):
        if mirror_dir is None:
            raise RuntimeError(
                'Google Drive resume requires a mounted Drive folder')
        resume_path = os.path.join(mirror_dir, 'latest.pt')
        if os.path.exists(resume_path):
            print(f'resuming {name} from {resume_path}')
        else:
            print(f'no Drive checkpoint for {name}; starting fresh')
            resume_path = None
    backbone, head = make_policy(name)
    history = train_action_head(
        head, backbone, train_loader, stats, tokenizer, device,
        total_steps=steps,
        warmup_steps=min(settings['WARMUP_STEPS'], steps - 1),
        log_every=settings['LOG_EVERY'],
        checkpoint_every=settings['CHECKPOINT_EVERY'],
        validate_every=settings['VALIDATE_EVERY'],
        validation_loader=validation_loader,
        checkpoint_dir=f'{CHECKPOINT_ROOT}/{name}',
        checkpoint_mirror_dir=mirror_dir,
        resume_from=resume_path,
        tensorboard_log_dir=(f'{TENSORBOARD_ROOT}/{name}'
                             if settings['RUN_MODE'] == 'full' else None))

    batch = next(iter(validation_loader))
    model_inputs = prepare_batch(batch, stats, device, backbone)
    target_bins, _ = action_targets(batch, stats, tokenizer, device)
    with torch.no_grad(), evaluation_mode(backbone), evaluation_mode(head):
        logits = action_head_logits(
            head, backbone, model_inputs, target_bins)
    validation_metrics = held_out_metrics(
        head, backbone, validation_loader, stats, tokenizer, device,
        max_batches=eval_batches)
    validation_ce = validation_metrics['loss']

    # Each head samples through its own inference path, so the AR draws
    # carry the conditioning the two parallel heads lack.
    sampled_grids = sample_action_grids(
        head, backbone, model_inputs, n_samples=samples)
    prediction = decode_action_chunk(
        head, backbone, model_inputs, tokenizer, stats,
        strategy='argmax').cpu()
    metrics = evaluate_open_loop(
        head, itertools.islice(validation_loader, eval_batches),
        tokenizer, stats, backbone, device)

    jitter = float(np.mean([
        temporal_jitter(grid) for grid in sampled_grids.cpu().numpy()
    ]))

    plot_training_curves({name: history})
    plt.show()

    joint_figure = plot_per_joint_metrics(
        history, SO101_ACTION_NAMES, title=head_label(name))
    joint_figure.savefig(
        f'{CHECKPOINT_ROOT}/{name}/per_joint_metrics.png')
    if mirror_dir is not None:
        joint_figure.savefig(
            os.path.join(mirror_dir, 'per_joint_metrics.png'))
    plt.show()

    mae_std = metrics['mae_in_standard_deviations'].nanmean().item()
    print(f'{name}: held-out CE={validation_ce:.3f} nats/token, '
          f'token accuracy={validation_metrics["accuracy"]:.1%}, '
          f'open-loop MAE={mae_std:.3f} training std, '
          f'jitter={jitter:.2f} bins/step')
    return {
        'head': head, 'backbone': backbone, 'history': history,
        'batch': batch, 'model_inputs': model_inputs,
        'target_bins': target_bins, 'validation_ce': validation_ce,
        'validation_metrics': validation_metrics,
        'open_loop_metrics': metrics,
        'mae_std': mae_std, 'jitter': jitter, 'prediction': prediction,
        'logits': logits.cpu(), 'samples': sampled_grids.cpu(),
    }

### Choose a run mode — edit this cell, then run the three trainers

Everything that controls the three training runs lives here, so you can
set it once and leave the trainer cells alone. In Colab these render as
form fields; edit and re-run **this cell only**, then run 4.5.1 to 4.5.3.

**sanity** runs 10 optimizer steps per head and prints the loss after
every step. Use it to check that the pipeline executes; its metrics mostly
reflect initialization. **full** uses 20,000 steps per head and keeps
restartable checkpoints. It also writes TensorBoard scalars and a static
loss-curve PNG.

For a full run, leave **Save to Google Drive** enabled. Colab asks for
Drive permission once, then every 1,000-step local checkpoint is copied
atomically under `MyDrive/<folder>/<run name>/<head>/`. To continue after
a disconnect, keep the same folder and run name, enable **Resume from
Google Drive**, and re-run this cell followed by the trainer cells.

In [ ]:
RUN_MODE = 'sanity'        #@param ['sanity', 'full']
GRID_SAMPLES = 64         #@param {type:"integer"}
AR_GRID_SAMPLES = 32      #@param {type:"integer"}
EVAL_BATCHES = 1          #@param {type:"integer"}
SAVE_TO_GOOGLE_DRIVE = True  #@param {type:"boolean"}
RESUME_FROM_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_FOLDER = 'lrm-book/ch04/checkpoints'  #@param {type:"string"}
DRIVE_RUN_NAME = 'chapter4-full-20k'  #@param {type:"string"}

MODES = {
    'sanity': dict(steps=10, warmup=5, log_every=1,
                   checkpoint_every=10, validate_every=2),
    'full': dict(steps=20_000, warmup=500, log_every=25,
                 checkpoint_every=1_000, validate_every=1_000),
}
assert RUN_MODE in MODES, "RUN_MODE must be 'sanity' or 'full'"
mode = MODES[RUN_MODE]
TRAIN_STEPS = mode['steps']
WARMUP_STEPS = mode['warmup']
LOG_EVERY = mode['log_every']
CHECKPOINT_EVERY = mode['checkpoint_every']
VALIDATE_EVERY = mode['validate_every']

if RESUME_FROM_GOOGLE_DRIVE and not SAVE_TO_GOOGLE_DRIVE:
    raise ValueError('Drive resume requires Drive checkpoint saving')
if RESUME_FROM_GOOGLE_DRIVE and RUN_MODE != 'full':
    raise ValueError('Drive resume is only available in full mode')
DRIVE_CHECKPOINT_ROOT = None
if RUN_MODE == 'full' and SAVE_TO_GOOGLE_DRIVE:
    if 'google.colab' not in sys.modules:
        print('Google Drive mirroring is available in Colab only.')
    else:
        from google.colab import drive
        drive.mount('/content/drive')
        drive_relative = os.path.join(
            DRIVE_FOLDER.strip('/'), DRIVE_RUN_NAME)
        DRIVE_CHECKPOINT_ROOT = os.path.join(
            '/content/drive/MyDrive', drive_relative)
        os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)

# Sanity logs all 10 updates. Full logs every 25 updates to both the
# notebook and TensorBoard without drawing a 20,000-point SVG.
# GRID_SAMPLES     complete action grids drawn per head for the coherence
#                  plots.
# AR_GRID_SAMPLES  the same for the autoregressive head, which decodes
#                  H x D positions in series -- keep it smaller.
# EVAL_BATCHES     held-out batches behind the open-loop MAE.

serial = ACTION_HORIZON * ACTION_DIM
print(f'run mode          : {RUN_MODE}')
print(f'steps per head    : {TRAIN_STEPS:,}')
print(f'loss log interval : every {LOG_EVERY} step(s)')
print(f'total steps       : {3 * TRAIN_STEPS:,} across three heads')
print(f'grid draws        : {GRID_SAMPLES} '
      f'(autoregressive: {AR_GRID_SAMPLES} x {serial} serial decodes)')
print(f'open-loop batches : {EVAL_BATCHES}')
print(f'Drive backup      : {DRIVE_CHECKPOINT_ROOT or "disabled"}')
if device.type == 'cpu':
    print('\nRunning on CPU. Full mode requires a GPU runtime:\n'
          'Runtime > Change runtime type > GPU.')

### 4.5.1 Factorized head — the one-shot baseline

The factorized head uses one learned slot per grid cell. Its projected
state is broadcast into every slot and read out independently. The
independence
assumption is visible in what is *absent*: no slot's output enters another
slot's computation.

In [ ]:
results = {}
results['factorized'] = run_head_experiment('factorized')

### 4.5.2 Autoregressive head — exact conditioning

Training is teacher-forced in one causal pass; inference feeds each
realized bin back through a KV cache. This head draws
`AR_GRID_SAMPLES` grids rather than `GRID_SAMPLES` because each draw
requires 96 serial decoding steps.

In [ ]:
results['autoregressive'] = run_head_experiment('autoregressive')

### 4.5.3 Parallel head — bidirectional action slots

Append `H` learned action positions to the Chapter 3 prefix. The prefix
stays causal while the action block attends
bidirectionally, and the whole grid comes back in one forward pass. The
decoding and execution examples below use this parallel policy.

In [ ]:
results['parallel'] = run_head_experiment('parallel')

# Use the parallel policy for the decoding and execution examples.
head = results['parallel']['head']
backbone = results['parallel']['backbone']
batch = results['parallel']['batch']
model_inputs = results['parallel']['model_inputs']

### Training curves, side by side

Three thin curves share one axis. Markers are held-out measurements; the
dotted line is `ln(256)`, the loss of a uniform policy. A curve pinned to
that line has not started learning; a training curve that falls while the
held-out markers do not is overfitting. Full mode also exposes the same
loss, accuracy, MAE, entropy, and learning-rate series through TensorBoard.

**Action-token accuracy** is exact argmax-bin accuracy over non-padded
action cells. Each joint has its own accuracy and MAE panel.

**MAE / training std** decodes each predicted bin to its centre, computes
`abs(predicted - demonstrated)` in raw action units, and divides by that
joint's training-set standard deviation. A value of 1.0 is an average
error of one training standard deviation. Autoregressive token metrics use
teacher forcing; its open-loop MAE uses serial generation. These metrics
measure held-out prediction, not closed-loop task success.

In [ ]:
training_figure = plot_training_curves(
    {name: results[name]['history'] for name in HEAD_NAMES})
training_curve_path = f'{CHECKPOINT_ROOT}/training_curves.png'
training_figure.savefig(training_curve_path)
if DRIVE_CHECKPOINT_ROOT is not None:
    training_figure.savefig(os.path.join(
        DRIVE_CHECKPOINT_ROOT, 'training_curves.png'))
plt.show()
print(f'saved static curves: {training_curve_path}')

if RUN_MODE == 'full':
    from tensorboard import notebook as tensorboard_notebook
    tensorboard_notebook.start(f'--logdir {TENSORBOARD_ROOT}')

## 4.6 What did it actually learn?

A falling loss says the policy puts more mass on expert bins. It does not
say whether a marginal kept both modes, or whether the cells go together.

### 4.6.1 Watching a softmax become bimodal

Fix one validation frame as an anchor, find the frames with the nearest
normalized proprioception, and plot their demonstrated bins above the
policy's marginals below. A bimodal **individual** softmax means genuine
ambiguity at that input; a bimodal **cluster mean** alone may just mean
the neighbours differ. Record the anchor, neighbour count, and seed with
each result. The figure footer stores all three values.

In [ ]:
from ch04.analysis import (collect_cell_softmaxes, collect_expert_pairs,
                           collect_joint_logit_mass, logit_mismatch_rates,
                           neighborhood_softmax_figure,
                           sampled_grids_by_head, set_seed)
from ch04.diagnostics import (plot_head_comparison,
                              plot_joint_logit_panels,
                              plot_temporal_traces)

SEED, ANCHOR_INDEX, N_NEIGHBORS = 0, 0, 32
BASE_JOINT, PAIR_DIMS, TIMESTEP = 0, (4, 5), 0
set_seed(SEED)

collected = collect_cell_softmaxes(
    head, backbone, validation_loader, stats, tokenizer, device,
    timestep=TIMESTEP, control=BASE_JOINT, max_batches=8)

# Figure 4.8
neighborhood_softmax_figure(
    collected, anchor_index=ANCHOR_INDEX,
    n_neighbors=min(N_NEIGHBORS, collected['states'].shape[0]),
    checkpoint=f'{CHECKPOINT_ROOT}/parallel/best.pt', seed=SEED)
plt.show()

### 4.6.2 Measuring joint mismatch

Pick two controls that move together in the demonstrations and compute
their pair mass directly
from the softmax logits. The held-out demonstrations get their own panel,
rather than being drawn over the policy heatmaps, so off-support mass is
visible without Monte Carlo noise or overlapping marks.

For the factorized and parallel heads this is the exact product of their
two categorical cell marginals, averaged over frames. The autoregressive
second-cell logits are teacher-forced on the demonstrated preceding bins.
This panel therefore shows a conditional slice rather than an enumeration
of every possible autoregressive prefix.

In [ ]:
set_seed(SEED)

masses, grids = {}, {}
for name in HEAD_NAMES:
    peer = {name: results[name]['head']}
    peer_backbone = results[name]['backbone']
    inputs = results[name]['model_inputs']
    masses[name] = collect_joint_logit_mass(
        peer[name], peer_backbone, validation_loader, stats, tokenizer,
        device, dims=PAIR_DIMS, timestep=TIMESTEP, max_batches=8)
    grids.update(sampled_grids_by_head(
        peer, peer_backbone, inputs, n_samples=12))

expert_pairs = collect_expert_pairs(
    validation_loader, stats, tokenizer, device, dims=PAIR_DIMS,
    timestep=TIMESTEP, max_batches=8)

# Figure 4.9
plot_joint_logit_panels(
    masses, expert_pairs, bin_range=(0, ACTION_BINS),
    dim_labels=(str(PAIR_DIMS[0]), str(PAIR_DIMS[1])))
plt.show()

rates = logit_mismatch_rates(
    masses, ACTION_BINS // 2, ACTION_BINS // 2)
for name in HEAD_NAMES:
    print(f'{name:>15}: {rates[name]:.1%} logit mass off-diagonal')

In [ ]:
# The same comparison along the temporal axis: independent per-cell
# sampling shows up as jitter between consecutive timesteps.
plot_temporal_traces(grids, control=PAIR_DIMS[0])
plt.show()

## 4.7 From distribution to motor commands

Argmax picks a learned **mode** rather than the mean of two modes. Per-cell
argmax can still assemble a combination the expert never demonstrated.
The tokenizer
returns normalized midpoints and Chapter 2's denormalizer converts those
to the dataset's raw command units.

In [ ]:
from ch04.analysis import decoded_chunk_stream, open_loop_episode_trace
from ch04.diagnostics import (plot_execution_schedules,
                              plot_open_loop_episode)
from ch04.execution import execution_schedules

# Figure 4.10: three schedules over one decoded chunk stream.
chunks = decoded_chunk_stream(
    head, backbone, validation_loader, tokenizer, stats, device,
    max_batches=8)
plot_execution_schedules(execution_schedules(chunks), control=0)
plt.show()

In [ ]:
# Figure 4.11: one non-overlaid six-control figure per trained head.
# Every variant sees the same ordered held-out frames. The policy output
# is never fed back, so separation from the expert is open-loop error.
open_loop_traces = {}
for name in HEAD_NAMES:
    result = results[name]
    trace = open_loop_episode_trace(
        result['head'], result['backbone'], validation_loader,
        tokenizer, stats, device, max_batches=8)
    open_loop_traces[name] = trace
    episode_figure = plot_open_loop_episode(
        trace['predicted'], trace['expert'], trace['valid'],
        joint_names=SO101_ACTION_NAMES, head_name=name)
    episode_path = (
        f'{CHECKPOINT_ROOT}/{name}/figure_4_11_open_loop_episode.png')
    episode_figure.savefig(episode_path)
    if DRIVE_CHECKPOINT_ROOT is not None:
        os.makedirs(
            os.path.join(DRIVE_CHECKPOINT_ROOT, name), exist_ok=True)
        episode_figure.savefig(os.path.join(
            DRIVE_CHECKPOINT_ROOT, name,
            'figure_4_11_open_loop_episode.png'))
    plt.show()

    # Padding-aware error matrix: [horizon offset, control].
    offset_mae = result['open_loop_metrics'][
        'mae_in_standard_deviations']
    per_control_mae = torch.nanmean(offset_mae, dim=0)
    summary = ', '.join(
        f'{joint}={value:.3f}' for joint, value in zip(
            SO101_ACTION_NAMES, per_control_mae.tolist()))
    print(f'{head_label(name)} mean MAE/std by control: {summary}')
    print(f'saved: {episode_path}')

### Export one chunk for a calibrated SO-101

This cell exports the parallel policy's first denormalized 16-step chunk,
including the training-data
range used for a safety check. Download the file and replay it from the
computer physically connected to the follower arm. The local command is
a dry run until `--execute` is supplied, and LeRobot caps each relative
joint target. Clear the workspace and keep an emergency stop ready.

The observation stays fixed while the chunk executes. Treat this motion
as a hardware smoke test, not a closed-loop task-success measurement.

In [ ]:
from ch04.so101 import export_action_chunk

SO101_CHUNK_PATH = f'{CHECKPOINT_ROOT}/parallel/so101_chunk.npz'
export_action_chunk(
    SO101_CHUNK_PATH, results['parallel']['prediction'][0].numpy(),
    fps=30, action_min=stats['action'].get('min'),
    action_max=stats['action'].get('max'),
    source=f'{CHECKPOINT_ROOT}/parallel/best.pt')
print(f'exported: {SO101_CHUNK_PATH}')
print('Local preview: ch04-so101-replay so101_chunk.npz '
      '--port <FOLLOWER_PORT>')
print('After inspection, repeat with --execute.')

## The three designs, side by side

**Held-out CE** is how well each head fits the expert distribution.
**Open-loop MAE** is decoded error in units of the training standard
deviation, so 1.0 means no better than predicting the mean. **Jitter** is
the mean step-to-step change in a sampled chunk — the cost of factorizing.

In `sanity` mode these bars compare initialization noise. Select `full`
and re-run before reading anything into them.

In [ ]:
plot_head_comparison({name: results[name] for name in HEAD_NAMES})
plt.show()

print(f"{'head':<16}{'held-out CE':>13}{'MAE / std':>12}{'jitter':>10}")
for name in HEAD_NAMES:
    result = results[name]
    print(f"{name:<16}{result['validation_ce']:>13.3f}"
          f"{result['mae_std']:>12.3f}{result['jitter']:>10.2f}")

## Where to go next

- **Train for real.** Set `RUN_MODE = 'full'`, or from a terminal:
  `ch04-train --head all --steps 20000 --tensorboard-dir runs/ch04`.
  Figures regenerate from a
  checkpoint with
  `ch04-figures checkpoints/parallel/best.pt --head parallel`.
- **Sweep the bin count.** Coarser bins raise quantization error; finer
  bins leave fewer examples per class. 256 is a default to validate per
  task, not a constant.
- **Try the decoding rules.** `decode_action_chunk(..., strategy='sample',
  temperature=..., top_p=...)` — temperature sets how peaked the
  distribution is, top-p decides how much of the tail survives.
- **Interpret the diagnostics correctly.** Open-loop agreement measures
  prediction error on held-out demonstrations, not closed-loop task
  success. Better sampling does not remove joint mismatch from independent
  categorical outputs; a joint generative action model is needed for that.